In [1]:
# 08_recall_stress_test.ipynb -- Difficulty-stratified recall/precision stress test, turning the compliance axis from a tautology into a measured frontier.
# Injects obfuscated PII variants (D1..D3) and distractor patterns (UUIDs, git SHAs, versions, timestamps) that look sensitive but are not.
# Uses the SAME detectors as notebook 02; measures recall (sensitive caught) and precision (not firing on distractors) per difficulty. No LLM calls.
# Output paths are intentionally not printed (anonymity).

import re
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
                     "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys"})
GRAYS = ["0.10", "0.45"]
MARKERS = ["o", "s"]

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
TAB, FIG = ROOT / "results" / "tables", ROOT / "results" / "figures"
for d in (TAB, FIG):
    d.mkdir(parents=True, exist_ok=True)

# --- Detectors identical to notebook 02 ------------------------------------
def luhn_ok(num):
    s = re.sub(r"\D", "", num)
    if len(s) != 16:
        return False
    total = sum(int(d) for d in s[-1::-2])
    for d in s[-2::-2]:
        d2 = int(d) * 2
        total += d2 - 9 if d2 > 9 else d2
    return total % 10 == 0

def rrn_ok(num):
    d = re.sub(r"\D", "", num)
    if len(d) != 13:
        return False
    w = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    return (11 - (sum(int(d[i]) * w[i] for i in range(12)) % 11)) % 10 == int(d[12])

PATTERNS = {
    "RRN": re.compile(r"\b\d{6}-?\d{7}\b"),
    "CARD": re.compile(r"\b(?:\d{4}[- ]?){3}\d{4}\b"),
    "SECRET": re.compile(r"\b(?:sk-[A-Za-z0-9]{20,}|AKIA[A-Z0-9]{16}|ghp_[A-Za-z0-9]{30,}|[0-9a-f]{40})\b"),
    "EMAIL": re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"),
    "PHONE": re.compile(r"\b01[016789]-?\d{3,4}-?\d{4}\b"),
    "ACCOUNT": re.compile(r"\b\d{11,14}\b"),
}
VALIDATORS = {"RRN": rrn_ok, "CARD": luhn_ok}

def detects(text):
    # True if ANY sensitive detector fires (validated) on text.
    for t, pat in PATTERNS.items():
        for m in pat.finditer(text):
            if t in VALIDATORS and not VALIDATORS[t](m.group(0)):
                continue
            return True
    return False

# --- Generators at graded difficulty ---------------------------------------
def _luhn_total(s):
    digits = [int(d) for d in s]
    total = sum(digits[-1::-2])
    for d in digits[-2::-2]:
        d2 = d * 2
        total += d2 - 9 if d2 > 9 else d2
    return total % 10

def gen_card_valid():
    p = "".join(str(random.randint(0, 9)) for _ in range(15))
    full = next(p + str(c) for c in range(10) if _luhn_total(p + str(c)) == 0)
    return full

def gen_rrn_valid():
    base = [random.randint(0, 9) for _ in range(12)]
    w = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    chk = (11 - (sum(b * ww for b, ww in zip(base, w)) % 11)) % 10
    d = "".join(map(str, base + [chk]))
    return d

def card_at(diff):
    c = gen_card_valid()
    if diff == "D0":   # canonical grouped
        return "-".join(c[i:i+4] for i in range(0, 16, 4))
    if diff == "D1":   # unusual separators / spaces
        return " ".join(c[i:i+4] for i in range(0, 16, 4))
    if diff == "D2":   # split across string concatenation
        return c[:8] + '" + "' + c[8:]
    return c[:4] + "\\n" + c[4:]  # D3: embedded newline escape

def rrn_at(diff):
    d = gen_rrn_valid()
    if diff == "D0":
        return d[:6] + "-" + d[6:]
    if diff == "D1":
        return d[:6] + " " + d[6:]
    if diff == "D2":
        return d[:6] + '"+"' + d[6:]
    return d[:3] + "." + d[3:6] + "-" + d[6:]

def email_at(diff):
    u = "".join(random.choice("abcdefghijklmnop") for _ in range(6))
    if diff == "D0":
        return f"{u}@example.com"
    if diff == "D1":
        return f"{u}+tag@example.co.kr"
    if diff == "D2":
        return f"{u} [at] example [dot] com"
    return f"{u}&#64;example.com"  # D3: html entity

DIFFS = ["D0", "D1", "D2", "D3"]
GEN = {"CARD": card_at, "RRN": rrn_at, "EMAIL": email_at}

# --- Distractors: look sensitive but are NOT (should NOT be masked) ---------
def distractors():
    out = []
    out.append(("UUID", "550e8400-e29b-41d4-a716-" + "".join(random.choice("0123456789abcdef") for _ in range(12))))
    out.append(("GITSHA", "".join(random.choice("0123456789abcdef") for _ in range(40))))  # 40-hex: collides with SECRET hex rule
    out.append(("VERSION", f"{random.randint(1,9)}.{random.randint(0,20)}.{random.randint(0,99)}"))
    out.append(("TIMESTAMP", f"{random.randint(1500000000,1700000000)}"))  # 10-digit epoch (not 11-14)
    out.append(("ORDERNO", "".join(str(random.randint(0,9)) for _ in range(random.choice([11,12,13])))))  # collides with ACCOUNT
    return out

N = 200

# --- Recall per difficulty (sensitive tokens that SHOULD be caught) --------
recall_rows = []
for diff in DIFFS:
    for typ, gen in GEN.items():
        caught = 0
        for _ in range(N):
            tok = gen(diff)
            if detects(tok):
                caught += 1
        recall_rows.append({"difficulty": diff, "type": typ, "recall": caught / N})
recall_df = pd.DataFrame(recall_rows)
recall_overall = recall_df.groupby("difficulty")["recall"].mean().reset_index()
recall_df.to_csv(TAB / "recall_by_difficulty_type.csv", index=False)
recall_overall.to_csv(TAB / "recall_by_difficulty.csv", index=False)

# --- Precision: fraction of distractors correctly NOT flagged --------------
# False positive if a sensitive detector fires on a distractor.
prec_rows = []
for name, _ in distractors():
    pass
fp = {}
tot = {}
for _ in range(N):
    for name, val in distractors():
        tot[name] = tot.get(name, 0) + 1
        if detects(val):
            fp[name] = fp.get(name, 0) + 1
for name in tot:
    prec_rows.append({"distractor": name, "false_positive_rate": fp.get(name, 0) / tot[name]})
prec_df = pd.DataFrame(prec_rows)
overall_fp = prec_df["false_positive_rate"].mean()
prec_df.to_csv(TAB / "distractor_false_positives.csv", index=False)

# --- Figure: recall vs difficulty (overall + per type) ---------------------
fig, ax = plt.subplots(figsize=(5.4, 3.5))
for i, typ in enumerate(["CARD", "RRN", "EMAIL"]):
    sub = recall_df[recall_df["type"] == typ]
    ax.plot(sub["difficulty"], sub["recall"], color=GRAYS[0] if i == 0 else str(0.25 + 0.2 * i),
            marker=MARKERS[0], linewidth=1.5, label=typ)
ax.plot(recall_overall["difficulty"], recall_overall["recall"], color="black", marker="D",
        linewidth=2.0, linestyle="--", label="mean")
ax.set_xlabel("Injection difficulty (D0 canonical -> D3 obfuscated)")
ax.set_ylabel("Masker recall")
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, fontsize=9, ncol=2)
save_fig = lambda n: (fig.savefig(FIG / f"{n}.png", dpi=600, bbox_inches="tight"),
                      fig.savefig(FIG / f"{n}.pdf", bbox_inches="tight"), plt.close(fig))
save_fig("fig_recall_stress")

print("Recall by difficulty (mean over types):")
print(recall_overall.to_string(index=False))
print("\nDistractor false-positive rates (precision stress):")
print(prec_df.to_string(index=False))
print(f"\nMean distractor false-positive rate: {overall_fp:.3f}")


Recall by difficulty (mean over types):
difficulty   recall
        D0 1.000000
        D1 0.666667
        D2 0.000000
        D3 0.001667

Distractor false-positive rates (precision stress):
distractor  false_positive_rate
      UUID                  0.0
    GITSHA                  1.0
   VERSION                  0.0
 TIMESTAMP                  0.0
   ORDERNO                  1.0

Mean distractor false-positive rate: 0.400
